# 03 · Servidor OPC UA com modelo de informação

**Objetivo:** ver a diferença entre "uma lista de tags" e um **modelo de informação**: objetos, tipos, unidades, faixas, permissões e métodos.

No Modbus o cliente precisa de uma planilha para saber que o registrador 0 é o nível. No OPC UA **o servidor descreve a si mesmo**: o cliente navega e descobre o que existe.

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from common.notebook import load, run_in_background, require_broker
import asyncio
from asyncua import Client, ua
srv = load("03-opcua-server-model/server.py")
URL = "opc.tcp://localhost:4841/iiot-examples/"
run_in_background(lambda: srv.main("opc.tcp://0.0.0.0:4841/iiot-examples/"), port=4841)

servidor ativo na porta 4841


## 1. Navegando no espaço de endereços

Tudo no OPC UA é um **nó** com um `NodeClass`: `Object` (uma coisa), `Variable` (um valor), `Method` (uma ação). O servidor organiza os nós numa árvore, igual pastas.

In [2]:
async def arvore(node, nivel=0, max_nivel=5):
    nome = (await node.read_browse_name()).Name
    classe = (await node.read_node_class()).name
    linha = "    " * nivel + f"{nome}  [{classe}]"
    if classe == "Variable":
        try:
            linha += f"  = {await node.read_value()!r}"
        except Exception:
            pass
    print(linha)
    if nivel < max_nivel and classe == "Object":
        for filho in await node.get_children(refs=ua.ObjectIds.HierarchicalReferences):
            if (await filho.read_node_class()) in (ua.NodeClass.Object, ua.NodeClass.Variable, ua.NodeClass.Method):
                await arvore(filho, nivel + 1, max_nivel)

async def main():
    async with Client(URL) as c:
        idx = await c.get_namespace_index(srv.NS_URI)
        print("namespace", srv.NS_URI, "-> índice", idx, "\n")
        plant = await c.nodes.objects.get_child(f"{idx}:Plant")
        await arvore(plant)

await main()

namespace urn:iiot-protocols-examples:plant -> índice 2 

Plant  [Object]
    Line1  [Object]
        Filler01  [Object]
            State  [Variable]  = 'IDLE'
            Speed  [Variable]  = 0.0
            GoodCount  [Variable]  = 0
            RejectCount  [Variable]  = 0
            Temperature  [Variable]  = 21.024997395914713
            Setpoints  [Object]
                TargetSpeed  [Variable]  = 400.0
            Start  [Method]
            Stop  [Method]
            ResetCounters  [Method]


**Interpretação**

* `Plant → Line1 → Filler01` é a hierarquia da planta. Um cliente que conecta pela primeira vez **descobre** a enchedora sem configuração prévia.
* `Speed` e `Temperature` têm filhos `EngineeringUnits` e `EURange`: **a unidade e a faixa viajam junto com o valor**.
* `Start`, `Stop` e `ResetCounters` aparecem como `Method`: ações que o servidor executa, não variáveis que o cliente escreve.

## 2. Metadados: tipo, unidade, faixa, permissão

In [3]:
async def metadados():
    async with Client(URL) as c:
        idx = await c.get_namespace_index(srv.NS_URI)
        f = await c.nodes.objects.get_child([f"{idx}:Plant", f"{idx}:Line1", f"{idx}:Filler01"])
        for nome in ["Speed", "Temperature", "GoodCount", "State"]:
            v = await f.get_child(f"{idx}:{nome}")
            tipo = (await c.get_node(await v.read_data_type()).read_browse_name()).Name
            acesso = await v.read_attribute(ua.AttributeIds.UserAccessLevel)
            gravavel = bool(acesso.Value.Value & ua.AccessLevel.CurrentWrite.mask)
            extra = ""
            try:
                eu = await (await v.get_child("0:EngineeringUnits")).read_value()
                r = await (await v.get_child("0:EURange")).read_value()
                extra = f"unidade {eu.DisplayName.Text}, faixa {r.Low}..{r.High}"
            except Exception:
                pass
            print(f"{nome:<12} tipo {tipo:<8} gravável={gravavel!s:<5} {extra}")
        sp = await f.get_child([f"{idx}:Setpoints", f"{idx}:TargetSpeed"])
        acesso = await sp.read_attribute(ua.AttributeIds.UserAccessLevel)
        print(f"{'TargetSpeed':<12} tipo Double   gravável={bool(acesso.Value.Value & ua.AccessLevel.CurrentWrite.mask)}")

await metadados()

Speed        tipo Double   gravável=False unidade bottles/min, faixa 0.0..600.0
Temperature  tipo Double   gravável=False unidade °C, faixa -20.0..120.0
GoodCount    tipo UInt32   gravável=False 
State        tipo String   gravável=False 
TargetSpeed  tipo Double   gravável=True


Só o **setpoint** é gravável. Medições (velocidade, contadores, temperatura) são só leitura: o servidor protege o processo de um cliente que tente "escrever uma velocidade". No Modbus essa proteção depende de cada fabricante; no OPC UA é um atributo padrão de todo nó.

## 3. Métodos com validação no servidor

In [4]:
async def metodos():
    async with Client(URL) as c:
        idx = await c.get_namespace_index(srv.NS_URI)
        f = await c.nodes.objects.get_child([f"{idx}:Plant", f"{idx}:Line1", f"{idx}:Filler01"])
        state = await f.get_child(f"{idx}:State")
        good = await f.get_child(f"{idx}:GoodCount")
        speed = await f.get_child(f"{idx}:Speed")
        print("estado inicial:", await state.read_value())
        print("Start()  ->", await f.call_method(f"{idx}:Start"))
        for _ in range(3):
            await asyncio.sleep(2)
            print(f"  estado {await state.read_value():<8} velocidade {await speed.read_value():6.1f} garrafas/min   boas {await good.read_value()}")
        print("ResetCounters() ->", await f.call_method(f"{idx}:ResetCounters"))
        await asyncio.sleep(1)
        print("  boas depois do reset:", await good.read_value())
        print("Stop()   ->", await f.call_method(f"{idx}:Stop"))
        await asyncio.sleep(2)
        print("  estado:", await state.read_value(), "| velocidade caindo:", round(await speed.read_value(), 1))
        try:
            await state.write_value(ua.Variant("FAULTED", ua.VariantType.String))
        except ua.UaStatusCodeError as e:
            print("\ntentar escrever State direto ->", type(e).__name__)

await metodos()

estado inicial: IDLE
Start()  -> ok


  estado RUNNING  velocidade  237.0 garrafas/min   boas 5


  estado RUNNING  velocidade  334.9 garrafas/min   boas 15


  estado RUNNING  velocidade  371.4 garrafas/min   boas 27
ResetCounters() -> ok


  boas depois do reset: 3
Stop()   -> ok


  estado: IDLE | velocidade caindo: 24.1

tentar escrever State direto -> BadUserAccessDenied


**Interpretação**

* `Start()` coloca a máquina em `RUNNING`; a velocidade sobe em rampa até o setpoint (400 garrafas/min) e o contador de boas cresce.
* `ResetCounters()` zera os contadores **dentro do servidor**: o cliente pede a ação, não escreve zero em cada variável.
* `Stop()` volta para `IDLE` e a velocidade cai.
* Escrever `State` direto é **recusado**: o estado da máquina só muda pelos métodos. Se a máquina estiver em `FAULTED`, o próprio `Start()` responde "rejected: reset the fault first". A regra de negócio fica junto do equipamento, não espalhada pelos clientes.

## Resumo

* OPC UA = **dados + significado**: hierarquia, tipos, unidades, faixas e permissões vêm do servidor.
* **Métodos** encapsulam ações com validação, em vez de o cliente escrever bits de comando.
* Esse modelo é o que permite a um MES ou historiador se conectar e entender a máquina sem planilha de tags.

**Próximo:** o notebook 04 mostra o lado do cliente e por que **subscription** é melhor que polling.